# Lecture 05 题目讲义：上机复习

定位：面向上机考前复习，按题型从稳分到进阶组织。

建议做法：先独立写，再对照后面的关键点与参考代码。

## Problem 1: 单链表反转

**题目**：给定单链表头结点，原地反转链表并返回新头结点。

**关键点**：先保存后继，再改指针；循环结束后返回 `prev`。

**复杂度**：时间 `O(n)`，空间 `O(1)`。

In [ ]:
class Node:
    def __init__(self, val, nxt=None):
        self.val = val
        self.next = nxt

def reverse_list(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
    return prev


## Problem 2: 多项式加法

**题目**：两个多项式按 `(coef, exp)` 给出，要求输出相加后的结果，幂次降序，系数为 0 的项不输出。

**关键点**：本质是按指数合并有序序列；指数相同时先合并，再判断是否保留。

**复杂度**：若输入已按指数排序，则可做到 `O(n + m)`。

In [ ]:
def add_poly(p1, p2):
    i = j = 0
    ans = []
    while i < len(p1) and j < len(p2):
        c1, e1 = p1[i]
        c2, e2 = p2[j]
        if e1 == e2:
            if c1 + c2 != 0:
                ans.append((c1 + c2, e1))
            i += 1
            j += 1
        elif e1 > e2:
            ans.append((c1, e1))
            i += 1
        else:
            ans.append((c2, e2))
            j += 1
    ans.extend(p1[i:])
    ans.extend(p2[j:])
    return ans


## Problem 3: KMP 首次匹配

**题目**：在文本串 `text` 中寻找模式串 `pattern` 第一次出现的位置，不存在返回 `-1`。

**关键点**：失配时让 `j` 跳到 `next[j-1]`；文本指针不回退。

**复杂度**：`O(n + m)`。

In [ ]:
def build_next(pattern):
    nxt = [0] * len(pattern)
    j = 0
    for i in range(1, len(pattern)):
        while j > 0 and pattern[i] != pattern[j]:
            j = nxt[j - 1]
        if pattern[i] == pattern[j]:
            j += 1
        nxt[i] = j
    return nxt

def kmp_find(text, pattern):
    if not pattern:
        return 0
    nxt = build_next(pattern)
    j = 0
    for i, ch in enumerate(text):
        while j > 0 and ch != pattern[j]:
            j = nxt[j - 1]
        if ch == pattern[j]:
            j += 1
        if j == len(pattern):
            return i - len(pattern) + 1
    return -1


## Problem 4: 字符串乘方

**题目**：给定字符串 `s`，求最大的 `k`，使得存在字符串 `a` 满足 `s = a^k`。

**关键点**：若 `L = len(s)`，`p = L - next[-1]`，且 `L % p == 0`，则答案为 `L // p`。

**复杂度**：`O(n)`。

In [ ]:
def string_power(s):
    if not s:
        return 0
    nxt = build_next(s)
    period = len(s) - nxt[-1]
    if period != len(s) and len(s) % period == 0:
        return len(s) // period
    return 1


## Problem 5: 二叉树层序遍历

**题目**：输出二叉树的层序遍历序列。

**关键点**：空树直接返回空列表；使用队列维护当前待访问结点。

**复杂度**：`O(n)`。

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def level_order(root):
    if root is None:
        return []
    q = deque([root])
    ans = []
    while q:
        node = q.popleft()
        ans.append(node.val)
        if node.left:
            q.append(node.left)
        if node.right:
            q.append(node.right)
    return ans


## Problem 6: 无权图最短步数

**题目**：给定无权图和起点终点，求最少经过多少条边可以到达终点；不可达返回 `-1`。

**关键点**：无权最短路优先想到 BFS；访问标记要在入队时完成。

**复杂度**：邻接表写法下为 `O(n + m)`。

In [ ]:
from collections import deque

def shortest_steps(graph, start, target):
    q = deque([(start, 0)])
    seen = {start}
    while q:
        u, dist = q.popleft()
        if u == target:
            return dist
        for v in graph[u]:
            if v not in seen:
                seen.add(v)
                q.append((v, dist + 1))
    return -1


## Problem 7: 等式与不等式判定

**题目**：给定若干约束，如 `x==y` 与 `x!=y`，判断这些约束能否同时成立。

**关键点**：先处理所有等式，构造等价类；再检查不等式是否冲突。

**复杂度**：并查集近似线性。

In [ ]:
class DSU:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        self.parent[self.find(x)] = self.find(y)

def can_satisfy(equations):
    dsu = DSU()
    for a, op, b in equations:
        if op == '==':
            dsu.union(a, b)
    for a, op, b in equations:
        if op == '!=' and dsu.find(a) == dsu.find(b):
            return False
    return True


## Problem 8: Top-k 最小堆

**题目**：数据流不断到来，实时维护当前最大的 `k` 个元素。

**关键点**：维护大小为 `k` 的最小堆；堆顶始终是当前第 `k` 大。

**复杂度**：每次更新 `O(log k)`。

In [ ]:
import heapq

def top_k_stream(nums, k):
    heap = []
    for x in nums:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)
    return sorted(heap, reverse=True)


## 最后检查表

- 看数据范围，先估复杂度。
- 题目翻译成“状态 + 操作 + 终止条件”。
- 模板写完先喂最小样例。
- 最后检查空输入、重复值、头尾边界、输出格式。